In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

from scipy.stats import norm
from scipy.stats import t

## t Distribution

In [ ]:
x = np.arange(-4, 4, 0.01)

plots.plot(x, norm.pdf(x, 0,1), lw=1);
plots.plot(x, t.pdf(x, 30), lw=1);
plots.plot(x, t.pdf(x, 20), lw=1);
plots.plot(x, t.pdf(x, 10), lw=1);
plots.plot(x, t.pdf(x, 5), lw=1);

## Smile Leniency Effect

In [ ]:
Smiles = Table.read_table('/home/jovyan/shared/Data/Smiles.csv')
Smiles

In [ ]:
Smiles.group('Group', np.mean)

In [ ]:
def difference_of_means(table, outcome, group_label):
    """Takes: name of table, column label of binary categorical variable,
    column label of group-label variable
    Returns: Difference of proportions of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(outcome, group_label)  
    
    # table containing counts for each group
    summary_table = reduced.group(group_label, np.mean)
    
    return summary_table.column(1).item(0) - summary_table.column(1).item(1)

In [ ]:
def one_simulated_difference(table, outcome, group_label):
    """Takes: name of table, column label of binary cateogorcial variable,
    column label of group-label variable
    Returns: Difference of proportions of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of outcome variable and shuffled labels
    shuffled_table = table.select(outcome).with_column(
        'Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, outcome, 'Shuffled Label')  

In [ ]:
obs_diff_mean = difference_of_means(Smiles, 'Leniency', 'Group')
obs_diff_mean

In [ ]:
differences = make_array()

for i in np.arange(10000):
    new_difference = one_simulated_difference(Smiles, 'Leniency', 'Group')
    differences = np.append(differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist()
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_mean, color='red', lw=2);

In [ ]:
p_value = np.mean(differences <= obs_diff_mean)
p_value

In [ ]:
np.std(differences)

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist()
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_mean, color='red', lw=2);

x1 = np.arange(-1.5, 1.501, 0.01)
plots.plot(x1, norm.pdf(x1, 0, np.std(differences)));

In [ ]:
sd_smile = np.std(Smiles.where('Group', 'smile').column('Leniency'))
sd_neutral = np.std(Smiles.where('Group', 'neutral').column('Leniency'))

n_smile = Smiles.where('Group', 'smile').num_rows
n_neutral = Smiles.where('Group', 'neutral').num_rows

sd_smile, n_smile, sd_neutral, n_neutral

In [ ]:
sd_diff = np.sqrt(sd_smile**2 / n_smile + sd_neutral**2 / n_neutral)
sd_diff

In [ ]:
t_score = obs_diff_mean / sd_diff
t_score 

In [ ]:
plots.plot(x, t.pdf(x, min(n_smile, n_neutral)-1));
x2 = np.arange(-4, t_score + 0.01, 0.01)
plots.fill_between(x2, t.pdf(x2, min(n_smile, n_neutral)-1), color = 'red');

In [ ]:
t.cdf(t_score, min(n_smile, n_neutral)-1)

In [ ]:
norm.cdf(t_score)

### Birth Weight by Smoking Status 

In [ ]:
births = Table.read_table('/home/jovyan/shared/Data/baby.csv')

obs_diff_mean_weight = difference_of_means(births, 'Birth Weight','Maternal Smoker')
obs_diff_mean_weight

In [ ]:
differences_weight = make_array()

for i in np.arange(1000):
    new_difference_weight = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences_weight = np.append(differences_weight, new_difference_weight)

In [ ]:
Table().with_column('Difference Between Group Means', differences_weight).hist()
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_mean_weight, color='red', lw=2);

p_value_weight = np.mean(differences_weight>= obs_diff_mean_weight)
p_value_weight

In [ ]:
sd_smoke = np.std(births.where('Maternal Smoker', True).column('Birth Weight'))
sd_no = np.std(births.where('Maternal Smoker', False).column('Birth Weight'))

n_smoke = births.where('Maternal Smoker', True).num_rows
n_no = births.where('Maternal Smoker', False).num_rows

sd_smoke, n_smoke, sd_no, n_no

In [ ]:
sd_diff_weight = np.sqrt(sd_smoke**2 / n_smoke + sd_no**2 / n_no)
sd_diff_weight

In [ ]:
t_score_weight = obs_diff_mean_weight / sd_diff_weight
t_score_weight

In [ ]:
1 - t.cdf(t_score_weight, min(n_smoke, n_no)-1)

In [ ]:
t.cdf(-1 * t_score_weight, min(n_smoke, n_no)-1)

In [ ]:
1 - norm.cdf(t_score_weight, 0, 1)

In [ ]:
norm.cdf(-1*t_score_weight, 0, 1)